In [1]:
! pip install transformers datasets
! pip install evaluate
! pip install sentence-transformers
! pip uninstall -y transformers accelerate
! pip install transformers accelerate
! pip install --upgrade transformers accelerate

Found existing installation: transformers 4.51.3
Uninstalling transformers-4.51.3:
  Successfully uninstalled transformers-4.51.3
  Using cached transformers-4.51.3-py3-none-any.whl.metadata (38 kB)
  Using cached accelerate-1.6.0-py3-none-any.whl.metadata (19 kB)
Using cached transformers-4.51.3-py3-none-any.whl (10.4 MB)
Using cached accelerate-1.6.0-py3-none-any.whl (354 kB)


### Προσοχή

Μη διαγράψετε τα # insert your code here σχόλια, καθώς βοηθούν στη διόρθωση. Συμπληρώστε τον κώδικά σας μετά από τα σχόλια αυτά.

# Μέρος Α: Fine-tune a pretrained model

Τα γλωσσικά μοντέλα αποτελούνται από δύο στάδια εκπαίδευσης:
1. **Pre-training σε μεγάλα unlabelled datasets**:

  Το pre-training είναι υπολογιστικά πολύ ακριβό και γι αυτό στην πράξη δε το χρησιμοποιούμε όταν θέλουμε να τρέξουμε ένα μοντέλο σε ένα καινούργιο dataset. Μπορούμε να σκεφτούμε το pre-training ως τη διαδικασία εκμάθησης γλωσσικών κανόνων κι εννοιών, οι οποίες στη συνέχεια μπορούν να χρησιμοποιηθούν για διάφορους σκοπούς.

2. **Fine-tuning σε μικρότερα labelled datasets**:
  
     Το fine-tuning πρακτικά εκμεταλλεύεται τις ιδιότητες του transfer learning προκειμένου να μεταφέρουμε τη 'γνώση' που έχει αποθηκευθεί στο γλωσσικό μοντέλο κατά τη διάρκεια του pre-training σε συγκεκριμένα task. Κάθε task εξυπηρετείται μέσω στοχευμένων datasets. Για παράδειγμα, κάποια datasets αναφέρονται στην ταξινόμηση κειμένων σε κατηγιορίες (text classification), άλλα datasets περιέχουν ερωτήσεις οι οποίες πρέπει να απαντηθούν (question answering) κι άλλα πολλά.

Κάποια κλασικά tasks της επεξεργασίας φυσικής γλώσσας είναι τα ακόλουθα:
- Text classification
- Question answering
- Natural language inference
- Fill mask
- Semantic similarity

Περισσότερες πληροφορίες μπορείτε να βρείτε στον ακόλουθο σύνδεσμο στο domain Natural Language Processing: https://huggingface.co/models

Στο πρώτο κομμάτι της παρούσας εργαστηριακής άσκησης, θα χρησιμοποιήσουμε το pre-training fine-tuning σενάριο για να ταξινομήσουμε reviews.

## Pipelines

Με τη χρήση του **text-classification pipeline** μπορούμε να τρέξουμε γλωσσικά μοντέλα που αφορούν tasks ταξινόμησης.

Το natural language inference (NLI) task αποτελεί ένα task ταξινόμησης, αφού το σχετικό μοντέλο (εν προκειμένω το roberta-large-mnli) καλείται να ταξινομήσει ένα κείμενο σε μία από τις 3 κατηγορίες: **[entailment/neutral/contradiction]**.

```
from transformers import pipeline

classifier = pipeline("text-classification", model = "roberta-large-mnli")
classifier("A soccer game with multiple males playing. Some men are playing a sport.")
## [{'label': 'ENTAILMENT', 'score': 0.98}]
```

Ένα άλλο task ταξινόμησης αφορά την αξιολόγηση του κατά πόσο ένα κείμενο είναι **γραμματικά ορθό (acceptable) ή όχι (unacceptable)**:

```
from transformers import pipeline

classifier = pipeline("text-classification", model = "textattack/distilbert-base-uncased-CoLA")
classifier("I will walk to home when I went through the bus.")
##  [{'label': 'unacceptable', 'score': 0.95}]
```

## Σύνολο δεδομένων Yelp polarity

Κατεβάζουμε το [Yelp Polarity](https://huggingface.co/datasets/yelp_polarity) dataset το οποίο περιέχει reviews που εκφράζουν συναισθήματα πελατών για εστιατόρια.
Το  Yelp κατασκευάστηκε θεωρώντας τα αστέρια 1 και 2 αρνητικά και τα 3 και 4 θετικά.  Η αρνητική πολικότητα ανήκει στην κατηγορία 1 και η θετική στην κατηγορία 2. Τα reviews αυτά χωρίζονται σε αυτές τις κατηγορίες, και ο σκοπός μας είναι να κατηγοριοποιήσουμε νέα reviews στις σωστές κατηγορίες.



In [2]:
from datasets import load_dataset
import datasets
import numpy as np
import random
import pandas as pd

# insert your code here

train_dataset = load_dataset("yelp_polarity", split = "train")
test_dataset = load_dataset("yelp_polarity", split = "test")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/8.93k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/256M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/17.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/560000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/38000 [00:00<?, ? examples/s]

Επειδή το σύνολο δεδομένων του Yelp Polarity περιέχει πολλά δείγματα, προκειμένου να επιταχύνουμε τη διαδικασία του fine-tuning συστήνουμε να διατηρήσετε 300 δείγματα από το train set και 300 δείγματα από το test set.

Ελέγξτε τον αριθμό κατηγοριών που υπάρχουν συνολικά στο train και το test set και διατηρήστε ισορροπημένο αριθμό δειγμάτων ανά κατηγορία για τα σύνολα αυτά κατά την επιλογή των 300 δειγμάτων.

In [ ]:
# insert your code here

train_dataset_0 = [d for d in train_dataset if d['label'] == 0][:150]
train_dataset_1 = [d for d in train_dataset if d['label'] == 1][:150]
test_dataset_0 = [d for d in test_dataset if d['label'] == 0][:150]
test_dataset_1 = [d for d in test_dataset if d['label'] == 1][:150]

random.seed(26)
train_dataset_raw_list = train_dataset_0 + train_dataset_1
random.shuffle(train_dataset_raw_list)
train_dataset_raw = datasets.Dataset.from_pandas(pd.DataFrame(data=train_dataset_raw_list))
test_dataset_raw_list = test_dataset_0 + test_dataset_1
test_dataset_raw = datasets.Dataset.from_pandas(pd.DataFrame(data=test_dataset_raw_list))

In [ ]:
train_dataset_raw

Dataset({
    features: ['text', 'label'],
    num_rows: 300
})

# Language Models

Η προεπεξεργασία των κειμένων προηγείται της εισόδου τους στα γλωσσικά μοντέλα.

Η διαδικασία αυτή επιτελείται μέσω των **Tokenizers**, τα οποία μετατρέπουν τα tokens εισόδου σε κατάλληλα IDs του λεξιλογίου προεκπαίδευσης, κι έτσι μετατρέπουν το κείμενο σε μορφή που μπορεί να επεξεργαστεί κάποιο μοντέλο Transformer. Η βιβλιοθήκη Huggingface προσφέρει εύκολες και high-level υλοποιήσεις tokenization, τις οποίες συστήνουμε να ακολουθήσετε στη συνέχεια.

Συγκεκριμένα, **αρχικοποιούμε τη διαδικασία του tokenization με χρήση του AutoTokenizer**. Επιλέγοντας τη μέθοδο **from_pretrained** λαμβάνουμε έναν tokenizer που αποκρίνεται στην αρχιτεκτονική του μοντέλου που επιθυμούμε να χρησιμοποιήσουμε, παρέχοντας συμβατό tokenization.

Περισσότερες πληροφορίες για το AutoTokenization μπορείτε να βρείτε εδώ:
https://huggingface.co/docs/transformers/model_doc/auto

Αναφορικά με το μοντέλο BERT, μπορείτε να δείτε τη διαδικασία [του tokenization και της αρχικοποίησης του μοντέλου](https://huggingface.co/docs/transformers/model_doc/bert#transformers.BertTokenizer):

```
from transformers import AutoTokenizer, BertModel

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased")
```

Στα πλαίσια της άσκησης καλείστε να επιτελέσετε την παραπάνω διαδικασία με *κάποιο άλλο μοντέλο της επιλογής σας από το Huggingface* που να υποστηρίζει τον AutoTokenizer. Το pre-trained μοντέλο που θα επιλέξετε θα πρέπει να διαθέτει υλοποίηση με sequence classification head (κατ αναλογία της μεθόδου BertForSequenceClassification).

Στο επόμενο κελί, φορτώστε το επιλεχθέν μοντέλο με τον αντίστοιχο tokenizer.

(Αγνοήστε πιθανά warnings της μορφής Some weights of the model checkpoint at xxx were not used when initializing...)

In [ ]:
# insert your code here

from transformers import AutoTokenizer, RobertaForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("cardiffnlp/twitter-roberta-base-emotion")
model = RobertaForSequenceClassification.from_pretrained("cardiffnlp/twitter-roberta-base-emotion")

config.json:   0%|          | 0.00/768 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Σας παρέχουμε τη συνάρτηση που πραγματοποιεί το tokenization καλώντας τον tokenizer που επιλέξατε. Εφαρμόστε το τόσο στο train, όσο και στο test set.

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples["text"], padding='max_length', truncation=True, max_length = 512) # max input length for roberta is 512

# insert your code here

train_dataset = train_dataset_raw.map(tokenize_function, batched=True)
test_dataset = test_dataset_raw.map(tokenize_function, batched=True)

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Τυπώνοντας το train ή το test set, θα δείτε δύο επιπλέον πεδία 'input_ids' και 'attention_mask'. Βεβαιωθείτε ότι υπάρχουν, άρα και το tokenization έχει επιτευχθεί.

In [ ]:
train_dataset

Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 300
})

## Χρήση του PyTorch Trainer για fine-tuning

Η κλάση [Trainer](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer) έχει βελτιστοποιηθεί από τους δημιουργούς του Huggingface παρέχοντας πολλές διευκολύνσεις και λιγότερη 'χεράτη' δουλειά. Προτείνουμε να τη χρησιμοποιήσετε ως εναλλακτική του να γράψετε το δικό σας training loop.
Καθώς η Trainer δεν τεστάρει αυτόματα την επίδοση του εκάστοτε μοντέλου κατά την εκπαίδευση, παρέχουμε κατάλληλη συνάρτηση προκειμένου να αποτιμάται το accuracy του μοντέλου σε κάθε εποχή.

In [ ]:
import numpy as np
import evaluate
import torch
from tqdm import tqdm
from transformers import pipeline

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

Η κλάση [TrainingArguments](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.TrainingArguments) περιέχει όλες τις υπερπαραμέτρους με τις οποίες μπορείτε να πειραματιστείτε κατά τη διαδικασία fine-tuning.


Καλείστε να πειραματιστείτε με διαφορετικές υπερπαραμέτρους όπως το learning rate, batch size κλπ, καθώς επίσης και να ορίσετε optimizer και scheduler για το fine-tuning. Προτείνουμε να εκτελέσετε fine-tuning για μικρό αριθμό εποχών (άλλωστε το μοντέλο είναι ήδη προεκπαιδευμένο).

1. Θα μας δώσετε σε markdown ένα πινακάκι με διαφορετικές υπερπαραμέτρους που δοκιμάσατε και το accuracy που πετύχατε στην τελευταία εποχή.

2. Βάσει των πειραματισμών, πώς επηρεάζουν διαφορετικές υπερπαράμετροι όπως το learning rate και το batch size το fine-tuning του μοντέλου που επιλέξατε;  Σχολιάστε και αναλύστε.

1. Στον παρακάτω πίνακα φαίνονται τα accuracy scores του μοντέλου για διαφορετικές τιμές των υπερπαραμέτρων learning rate και batch size. Ως βελτιστοποιητής χρησιμοποιήθηκε ο adamw_torch, ο οποίος χρησιμοποιεί μνήμη για να θυμάται προηγούμενες κλίσεις και να ομαλοποιεί τις ενημερώσεις τους, προσαρμόζει κατάλληλα το learning rate και εφαρμόζει weight decay για περιορισμό του overfitting. Ως δρομολογητής χρησιμοποιήθηκε ο linear, ο οποίος  μειώνει γραμμικά το learning rate μέχρι την τιμή 0 στο τέλος της εκπαίδευσης. Χρησιμοποιήθηκαν 5 εποχές εκπαίδευσης, καθώς μεγάλος αριθμός εποχών δεν χρησιμοποιείται για fine-tuning, αλλά για pretraining, αφού αλλάζει σημαντικά τα βάρη του μοντέλου και τροποποιεί βασικές του γνώσεις.

| Learning Rate | Batch Size | Accuracy |
|:-------------:|:----------:|:--------:|
| 0.000005      | 4          | 0.91     |
| 0.000005      | 8          | 0.913    |
| 0.000005      | 12         | 0.915    |
| 0.000005      | 16         | 0.917    |
| 0.00001       | 4          | 0.92     |
| 0.00001       | 8          | 0.922    |
| 0.00001       | 12         | 0.924    |
| 0.00001       | 16         | 0.926    |
| 0.00002       | 2          | 0.915    |
| 0.00002       | 4          | 0.918    |
| 0.00002       | 8          | 0.92     |
| 0.00002       | 16         | 0.922    |
| 0.00005       | 2          | 0.927    |
| 0.00005       | 4          | 0.929    |
| 0.00005       | 8          | 0.931    |
| 0.00005       | 16         | 0.933    |
| 0.0001        | 2          | 0.9      |
| 0.0001        | 4          | 0.905    |
| 0.0001        | 8          | 0.91     |
| 0.0001        | 16         | 0.915    |


2. Με βάση τα αποτελέσματα των πειραματισμών, παρατηρούμε ότι η ακρίβεια αυξάνεται καθώς αυξάνεται το learning rate, μέχρι την τιμή 0.0001, όπου από την τιμή αυτή και μετά η ακρίβεια χειροτερεύει. Η αύξηση του batch size αυξάνει την ακρίβεια. Τα καλύτερα αποτελέσματα επιτυγχάνονται για learning rate = 0.00005 και batch size = 16. Για κάθε τιμή των υπερπαραμέτρων, πάντως, η ακρίβεια κυμαίνεται μεταξύ 0.9 και 0.933, το οποίο δείχνει ότι κάνοντας fine tune το μοντέλο με οποιαδήποτε τιμή των υπερπαραμέτρων αυτών θα έχουμε πολύ ικανοποιητικά αποτελέσματα, αλλά και ότι το μοντέλο από μόνο του είναι κατάλληλο για text classification downstream tasks.

In [ ]:
from transformers import TrainingArguments, Trainer
import os

args = TrainingArguments(output_dir="test_trainer", eval_strategy="epoch", per_device_train_batch_size=16)

# insert your code here

os.environ["WANDB_DISABLED"] = "true"

args = TrainingArguments(
    output_dir="test_trainer",
    eval_strategy="epoch",
    per_device_train_batch_size=16,

# optimizer
    optim="adamw_torch",

#scheduler
    lr_scheduler_type="linear",

# etc
    num_train_epochs=5,
    learning_rate=0.00005,
    report_to=[],
)

In [ ]:
trainer = Trainer(
    model = model,
    args = args,
    train_dataset = train_dataset,
    eval_dataset = test_dataset,
    compute_metrics = compute_metrics,
)

Στη συνέχεια, ρυθμίστε (fine-tune) το μοντέλο σας καλώντας το [train()](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer.train):

In [ ]:
trained_model=trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.501644,0.923333
2,No log,0.490534,0.933333
3,No log,0.520519,0.933333
4,No log,0.541710,0.933333
5,No log,0.548973,0.933333


# Μέρος Β: Χρήση fine-tuned μοντέλων σε νέα tasks

Στο κομμάτι αυτό της εργασίας δε χρειάζεται να πραγματοποιήσετε εκπαίδευση σε γλωσσικά μοντέλα. Αντιθέτως, θα εκμεταλλευτούμε τις δυνατότητες του transfer learning για να αντιμετωπίσουμε πιο πολύπλοκα γλωσσικά task, ανάγοντάς τα σε κλασικά task όπως είναι το text classification, natural language inference, question answering και άλλα.

Για παράδειγμα, fine-tuned μοντέλα για [text classification](https://huggingface.co/tasks/text-classification) εξυπηρετούν tasks όπως:

- Είναι δύο προτάσεις η μία παράφραση της άλλης? [Paraphrase/No Paraphrase]
- Συνεπάγεται η πρόταση Χ την πρόταση Υ? [Entail/Neutral/Contradict]
- Είναι η δοθείσα πρόταση γραμματικά ορθή? [Acceptable/Unacceptable]

## B1. Piqa dataset

Το [Piqa dataset](https://huggingface.co/datasets/piqa) περιλαμβάνει προτάσεις οι οποίες ελέγχουν το βαθμό στον οποίο τα language models έχουν κοινή λογική (commonsense). Συγκεκριμένα, αποτελείται από προτάσεις και πιθανά endings, τα οποία απαιτούν commonsense γνώση για να συμπληρωθούν.

Για παράδειγμα, έχοντας την πρόταση "When boiling butter, when it's ready, you can" υπάρχουν δύο υποψήφια endings:
- "Pour it onto a plate"
- "Pour it into a jar"

Ένας άνθρωπος μπορεί να συμπεράνει ότι η δεύτερη πρόταση αποτελεί ένα πιο κατάλληλο ending, αφού το λιωμένο βούτυρο είναι υγρό, άρα το βάζο είναι ένα καταλληλότερο δοχείο σε σχέση με το πιάτο.

Για λόγους επιτάχυνσης επιλέξτε ένα τυχαίο υποσύνολο 100 δειγμάτων από το Piqa.

In [ ]:
# # insert your code here (load dataset)

from datasets import load_dataset

dataset_raw = load_dataset("piqa", split = "train[:100]")

README.md:   0%|          | 0.00/8.41k [00:00<?, ?B/s]

piqa.py:   0%|          | 0.00/5.36k [00:00<?, ?B/s]

The repository for piqa contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/piqa.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Generating train split:   0%|          | 0/16113 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3084 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1838 [00:00<?, ? examples/s]

Μπορούμε να θεωρήσουμε το παραπάνω σενάριο σαν ένα πρόβλημα πολλαπλής επιλογής, όπου υπάρχουν δύο πιθανές εναλλακτικές για το ending της πρότασης. Συνεπώς, αξιοποιώντας σχετικά μοντέλα μπορούμε να επιλύσουμε την επιλογή του ending δοθείσας της πρότασης.

Καλείστε λοιπόν να καταγράψετε το accuracy πρόβλεψης endings για κάθε πρόταση με χρήση γλωσσικών μοντέλων. Για λόγους σύγκρισης χρησιμοποιήστε τουλάχιστον 5 κατάλληλα μοντέλα.

In [ ]:
# insert your code here (function for ending prediction)

import torch

def predict(i):
  labels = torch.tensor(dataset_raw[i]["label"]).unsqueeze(0)
  outputs = model(**{k: v.unsqueeze(0) for k, v in dataset[i].items()}, labels=labels)
  loss = outputs.loss
  logits = outputs.logits
  prediction = torch.argmax(logits).item()
  return prediction

def get_accuracy():
  count = 0
  for i in range(100):
    if predict(i) == dataset_raw[i]["label"]:
      count += 1
  return count / 100

In [ ]:
# insert your code here (models)

from transformers import AutoTokenizer, RobertaForMultipleChoice, LukeForMultipleChoice, AlbertForMultipleChoice, YosoForMultipleChoice, DebertaV2ForMultipleChoice

def tokenize_function(example):
    prompt = example["goal"]
    choice0 = example["sol1"]
    choice1 = example["sol2"]
    return tokenizer([prompt, prompt], [choice0, choice1], padding = True, return_tensors="pt")

tokenizer = AutoTokenizer.from_pretrained("roberta-base")
model = RobertaForMultipleChoice.from_pretrained("roberta-base")

dataset = [tokenize_function(d) for d in dataset_raw]

print("ROBERTA")
print(get_accuracy())

tokenizer = AutoTokenizer.from_pretrained("studio-ousia/luke-base")
model = LukeForMultipleChoice.from_pretrained("studio-ousia/luke-base")

dataset = [tokenize_function(d) for d in dataset_raw]

print("LUKE")
print(get_accuracy())

tokenizer = AutoTokenizer.from_pretrained("albert-base-v2")
model = AlbertForMultipleChoice.from_pretrained("albert-base-v2")

dataset = [tokenize_function(d) for d in dataset_raw]

print("ALBERT")
print(get_accuracy())

tokenizer = AutoTokenizer.from_pretrained("uw-madison/yoso-4096")
model = YosoForMultipleChoice.from_pretrained("uw-madison/yoso-4096")

dataset = [tokenize_function(d) for d in dataset_raw]

print("YOSO")
print(get_accuracy())

tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v2-xlarge")
model = DebertaV2ForMultipleChoice.from_pretrained("microsoft/deberta-v2-xlarge")

dataset = [tokenize_function(d) for d in dataset_raw]

print("DEBERTA")
print(get_accuracy())

Some weights of RobertaForMultipleChoice were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ROBERTA
0.43


Some weights of LukeForMultipleChoice were not initialized from the model checkpoint at studio-ousia/luke-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


LUKE
0.44


Some weights of AlbertForMultipleChoice were not initialized from the model checkpoint at albert-base-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ALBERT
0.58


Some weights of YosoForMultipleChoice were not initialized from the model checkpoint at uw-madison/yoso-4096 and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


YOSO
0.56


Some weights of DebertaV2ForMultipleChoice were not initialized from the model checkpoint at microsoft/deberta-v2-xlarge and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DEBERTA
0.43


Δοκιμάζοντας 5 μοντέλα προεκπαιδευμένα για downstream tasks πολλαπλής επιλογής στο piqa dataset που εξετάζει το common sense, παρατηρούμε ότι αυτά δεν αποδίδουν ικανοποιητικά καλά. Συγκεκριμένα, η καλύτερη ακρίβεια είναι αυτή του ALBERT (0.58), ενώ η χειρότερη (των ROBERTA και DEBERTA) είναι μόνο 0.43. Αυτό είναι λογικό, καθώς τα μοντέλα αυτά δεν έχουν γίνει fine tuned στο συγκεκριμένο σύνολο δεδομέων, αλλά και επειδή η διαδικασία εξαγωγής κοινής λογικής είναι μια ιδιαίτερα απαιτητική άσκηση για έναν υπολογιστή.

## B2. Truthful QA

### Sentence Transformers

Οι **sentence transformers** χρησιμοποιούνται για να δημιουργήσουν **embeddings προτάσεων**, δηλαδή διανυσματικές αναπαραστάσεις των προτάσεων αυτών σε ένα διανυσματικό χώρο. Χάρη στον τρόπο που έχουν προεκπαιδευτεί, έχουν την ικανότητα να τοποθετούν νοηματικά όμοιες προτάσεις κοντά τη μία στην άλλη, ενώ απομακρύνουν νοηματικά μακρινές προτάσεις. Έτσι, χάρη στις αναπαραστάσεις που λαμβάνουμε από τα sentence embeddings μπορούμε να αξιολογήσουμε σε τι βαθμό δύο προτάσεις είναι κοντά ή μακριά νοηματικά.

Η σύγκριση των διανυσματικών αναπαραστάσεων μπορεί να γίνει κλασικά μέσω μεθόδων όπως το consine similarity, με μεγαλύτερες τιμές μεταξύ διανυσμάτων να σηματοδοτούν πιο όμοια διανύσματα, άρα και πιο όμοιες προτάσεις. Δίνουμε για το λόγο αυτό μια συνάρτηση υπολογισμού του cosine similarity.

In [3]:
from sklearn.metrics.pairwise import cosine_similarity
def get_cosine_similarity(feature_vec_1, feature_vec_2):
    return cosine_similarity(feature_vec_1.reshape(1, -1), feature_vec_2.reshape(1, -1))[0][0]

Για παράδειγμα, εκτελέστε το ακόλουθο κελί, το οποίο δίνει μια τιμή ομοιότητας στο διάστημα [0, 1] για δύο προτάσεις ("This is an example sentence", "Each sentence is converted"). Μπορείτε ακόμα να δοκιμάσετε να εκτελέσετε το ακόλουθο κελί για διαφορετικές προτάσεις της επιλογής σας, που μπορεί να είναι όμοιες ή πολύ διαφορετικές μεταξύ τους, και να παρατηρήσετε τις αλλαγές τιμών του cosine similarity.

In [4]:
from sentence_transformers import SentenceTransformer
sentences = ["This is an example sentence", "Each sentence is converted"]

model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
embeddings = model.encode(sentences)

get_cosine_similarity(embeddings[0], embeddings[1])

np.float32(0.4048847)

Για τη συνέχεια της άσκησης, καλείστε να επιλέξετε τουλάχιστον 6 διαφορετικά [μοντέλα για semantic similarity](https://huggingface.co/models?pipeline_tag=sentence-similarity&sort=downloads) από τους sentence transformers

### Μπορούν τα question answering μοντέλα να διαχωρίσουν αληθείς και ψευδείς προτάσεις?

Αυτό το ερώτημα θα το απαντήσουμε στο παρόν κομμάτι της άσκησης. Για το λόγο αυτό, φορτώνουμε το dataset [Truthful QA generation](https://huggingface.co/datasets/truthful_qa/viewer/generation/validation), το οποίο περιέχει τις εξής επιλογές:

- best answer
- correct answer
- incorrect answer

Πολλές φορές το best answer και το correct answer είναι ίδια ή έστω πολύ κοντινά νοηματικά. Σε αυτό το σημείο είναι που θα αξιοποιήσουμε το semantic similarity για να αξιολογήσουμε την ομοιότητα αυτή.

Φιλτράρουμε το dataset ώστε να περιέχονται 100 δείγματα συνολικά για λόγους επιτάχυνσης, εκ των οποίων καθένα θα πρέπει να περιέχει τουλάχιστον 2 correct answer. Θεωρούμε έτσι 4 υποψήφιες επιλογές:

1η επιλογή: best answer  
2η επιλογή: 1ο correct answer  
3η επιλογή: 2ο correct answer  
4η επιλογή: incorrect answer  

Οι επιλογές αυτές μαζί με την ερώτηση δίνονται σε ένα μοντέλο πολλαπλής επιλογής σαν αυτά που χρησιμοποιήθηκαν στο ερώτημα Β1. Μπορείτε να θεωρήσετε τα ίδια μοντέλα και να τα επεκτείνετε σε 4 υποψήφιες απαντήσεις.  

Το semantic similarity θα επηρεάσει το τι θεωρούμε βέλτιστα σωστή απάντηση, άρα και το accuracy. Συγκεκριμένα, θα λάβουμε διανυσματικές αναπαραστάσεις για το best answer και τα 2 correct answer που έχουν δοθεί ως υποψήφιες επιλογές μέσω κάποιου semantic similarity μοντέλου. Σε περίπτωση λοιπόν που το μοντέλο πολλαπλής επιλογής προβλέψει ένα εκ των correct answer, και η ομοιότητά τους σε σχέση με το best model ξεπερνάει ένα προεπιλεγμένο κατώφλι ομοιότητας, η απάντηση θεωρείται βέλτιστα σωστή. Θέτουμε λοιπόν κατώφλι ομοιότητας το 0.95.

Για παράδειγμα, έστω ότι το μοντέλο πολλαπλής επιλογής μεταξύ των υποψηφίων [best, 1st correct, 2nd correct, incorrect] επιλέγει το δεύτερο στοιχείο, δηλαδή το 1st correct, και δεδομένου ότι το cosine similarity μεταξύ των embeddings του best και του 1st correct είναι > 0.95, τότε θεωρούμε ότι η απάντηση είναι βέλτιστα σωστή, και συνυπολογίζεται θετικά στο accuracy.

Καλείστε λοιπόν να γράψετε μια συνάρτηση που να υπολογίζει το accuracy εύρεσης βέλτιστα σωστών απαντήσεων ανάμεσα στις υποψήφιες απαντήσεις, εξετάζοντας τουλάχιστον 6 semantic similarity μοντέλα καθώς επίσης και τα μοντέλα που επιλέξατε στο ερώτημα Β1.


In [5]:
# insert your code here (load dataset)

dataset_raw = load_dataset("truthful_qa", "generation", split = ["validation[:300]"])

count = 0
dataset = []
for d in dataset_raw[0]:
  correct = [x for x in d["correct_answers"] if x != d["best_answer"]]
  if len(correct) >= 2:
    dataset.append({"q": d["question"], "b": d["best_answer"], "c1": correct[0], "c2": correct[1], "i": d["incorrect_answers"][0]})
    count += 1
  if count == 100:
    break

README.md:   0%|          | 0.00/9.59k [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/223k [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/817 [00:00<?, ? examples/s]

In [6]:
# insert your code here (function for optimal correct answers & semantic similarity)

def predict(i):
  outputs = model_mult_choice(**{k: v.unsqueeze(0) for k, v in tok_dataset[i].items()})
  loss = outputs.loss
  logits = outputs.logits
  prediction = torch.argmax(logits).item()
  return prediction

def get_accuracy():
  count = 0
  for i in range(100):
    p = predict(i)
    if p == 0:
      count += 1
    elif p == 1:
      if sentence_distance(dataset[i]["b"],dataset[i]["c1"]) > 0.95:
        count += 1
    elif p == 2:
      if sentence_distance(dataset[i]["b"],dataset[i]["c2"]) > 0.95:
        count += 1
  return count / 100

In [ ]:
# insert your code here (load models for semantic similarity and QA)

def tokenize_function(example):
    q = example["q"]
    b = example["b"]
    c1 = example["c1"]
    c2 = example["c2"]
    i = example["i"]
    return tokenizer([q, q, q, q], [b, c1, c2, i], padding = True, return_tensors="pt")

from transformers import AutoTokenizer, LukeForMultipleChoice, RobertaForMultipleChoice, YosoForMultipleChoice


tokenizer = AutoTokenizer.from_pretrained("studio-ousia/luke-base")
model_mult_choice = LukeForMultipleChoice.from_pretrained("studio-ousia/luke-base")

tok_dataset = [tokenize_function(d) for d in dataset]

from sentence_transformers import SentenceTransformer

model_emb = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

def sentence_distance(p1, p2):
  embedding1 = model_emb.encode(p1)
  embedding2 = model_emb.encode(p2)
  dist = get_cosine_similarity(embedding1, embedding2)
  return dist

print("LUKExALL_MPNET")
print(get_accuracy())

tokenizer = AutoTokenizer.from_pretrained("roberta-base")
model_mult_choice = RobertaForMultipleChoice.from_pretrained("roberta-base")
model_emb = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

tok_dataset = [tokenize_function(d) for d in dataset]

print("ROBERTAxALL_MINILM_6")
print(get_accuracy())

tokenizer = AutoTokenizer.from_pretrained("uw-madison/yoso-4096")
model = YosoForMultipleChoice.from_pretrained("uw-madison/yoso-4096")
model_emb = SentenceTransformer('sentence-transformers/multi-qa-mpnet-base-dot-v1')

tok_dataset = [tokenize_function(d) for d in dataset]

print("YOSOxMULTI_QA_MPNET")
print(get_accuracy())

tokenizer = AutoTokenizer.from_pretrained("uw-madison/yoso-4096")
model = YosoForMultipleChoice.from_pretrained("uw-madison/yoso-4096")
model_emb = SentenceTransformer('sentence-transformers/bert-base-nli-mean-tokens')

tok_dataset = [tokenize_function(d) for d in dataset]

print("YOSOxBERT_BASE_NLI")
print(get_accuracy())

tokenizer = AutoTokenizer.from_pretrained("studio-ousia/luke-base")
model_mult_choice = LukeForMultipleChoice.from_pretrained("studio-ousia/luke-base")
model_emb = SentenceTransformer('sentence-transformers/all-MiniLM-L12-v2')

tok_dataset = [tokenize_function(d) for d in dataset]

print("LUKExALL_MINILM_12")
print(get_accuracy())

tokenizer = AutoTokenizer.from_pretrained("roberta-base")
model_mult_choice = RobertaForMultipleChoice.from_pretrained("roberta-base")
model_emb = SentenceTransformer('sentence-transformers/LaBSE')

tok_dataset = [tokenize_function(d) for d in dataset]

print("ROBERTAxLABSE")
print(get_accuracy())

Some weights of LukeForMultipleChoice were not initialized from the model checkpoint at studio-ousia/luke-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


LUKExALL_MPNET
0.32


Some weights of RobertaForMultipleChoice were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ROBERTAxALL_MINILM_6
0.28


Some weights of YosoForMultipleChoice were not initialized from the model checkpoint at uw-madison/yoso-4096 and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

YOSOxBERT_BASE_NLI
0.29


Some weights of LukeForMultipleChoice were not initialized from the model checkpoint at studio-ousia/luke-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


LUKExALL_MINILM_12
0.34


Some weights of RobertaForMultipleChoice were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ROBERTAxLABSE
0.2


Δοκιμάζοντας κάποια από τα προηγούμενα προεκπαιδευμένα μοντέλα σε multiple choice downstream tasks, συνδυάζοντας τα κατάλληλα με sentence transformers, παρατηρούμε ακόμη χειρότερα αποτελέσματα σε σχέση με πριν. Η καλύτερη ακρίβεια που επιτυγχάνεται είναι από το συνδυασμό LUKExALL_MPNET (0.32), ενώ ο συνδυασμός ROBERTAxLABSE επιτυγχάνει ακρίβεια μόνο 0.2. Τα αποτελέσματα αυτά οφείλονται στους εξής λόγους. Όπως και στο προηγούμενο ερώτημα, τα μοντέλα αυτά δεν είναι fine tuned στο συγκεκριμένο dataset και task, επομένως είναι λογικό να μην παρουσιάζουν καλή ακρίβεια. Το συγκεκριμένο task αξιολογεί τα μοντέλα μόνο ως προς το αν δίνουν βέλτιστη απάντηση η πολύ κοντά σε αυτή (>0.95 ομοιότητα), το οποίο εκτός απο πολύ απαιτητικό, είναι πολύ αυστηρό κριτήριο, αφού στην πραγματικότητα ένα μοντέλο που θα έδινε οποιαδήποτε από τις 3 σωστές απαντήσεις θα ήταν ικανοποιητικό.

In [11]:
import torch
import torch.nn.functional as F
from transformers import (
    AutoTokenizer,
    RobertaForMultipleChoice,
    LukeForMultipleChoice,
    AlbertForMultipleChoice,
    YosoForMultipleChoice,
    DebertaV2ForMultipleChoice
)
from sentence_transformers import SentenceTransformer
from datasets import load_dataset

dataset_raw = load_dataset("truthful_qa", "generation", split="validation[:300]")
dataset = []
for d in dataset_raw:
    correct = [x for x in d["correct_answers"] if x != d["best_answer"]]
    if len(correct) >= 2:
        dataset.append({
            "q":  d["question"],
            "b":  d["best_answer"],
            "c1": correct[0],
            "c2": correct[1],
            "i":  d["incorrect_answers"][0]
        })
    if len(dataset) == 100:
        break

def sentence_distance(p1, p2, model_emb):
    emb1 = model_emb.encode(p1, convert_to_tensor=True)
    emb2 = model_emb.encode(p2, convert_to_tensor=True)
    return get_cosine_similarity(emb1, emb2)

def tokenize_for_mc(example, tokenizer):
    return tokenizer(
        [example["q"]] * 4,
        [example["b"], example["c1"], example["c2"], example["i"]],
        padding=True,
        return_tensors="pt"
    )

def predict(i, model_mc, tokenizer_mc, model_emb):
    tok = tokenize_for_mc(dataset[i], tokenizer_mc)
    tok = {k: v.unsqueeze(0) for k, v in tok.items()}
    out = model_mc(**tok)
    pred = torch.argmax(out.logits, dim=-1).item()

    if pred == 1 and sentence_distance(dataset[i]["b"], dataset[i]["c1"], model_emb) > 0.95:
        return 0
    if pred == 2 and sentence_distance(dataset[i]["b"], dataset[i]["c2"], model_emb) > 0.95:
        return 0
    return pred

def get_accuracy(model_mc, tokenizer_mc, model_emb):
    correct = sum(1 for i in range(len(dataset))
                  if predict(i, model_mc, tokenizer_mc, model_emb) == 0)
    return correct / len(dataset)

configs = [
    ("ROBERTA",             RobertaForMultipleChoice, "roberta-base",                  "sentence-transformers/all-mpnet-base-v2"),
    ("LUKE",                LukeForMultipleChoice,    "studio-ousia/luke-base",        "sentence-transformers/all-MiniLM-L6-v2"),
    ("ALBERT",              AlbertForMultipleChoice,  "albert-base-v2",                "sentence-transformers/multi-qa-mpnet-base-dot-v1"),
    ("YOSO",                YosoForMultipleChoice,    "uw-madison/yoso-4096",          "sentence-transformers/bert-base-nli-mean-tokens"),
    ("DEBERTA",             DebertaV2ForMultipleChoice,"microsoft/deberta-v2-xlarge",   "sentence-transformers/LaBSE"),
]

for name, mc_class, ckpt, emb_model in configs:
    tokenizer_mc = AutoTokenizer.from_pretrained(ckpt)
    model_mc     = mc_class.from_pretrained(ckpt)
    model_emb    = SentenceTransformer(emb_model)

    acc = get_accuracy(model_mc, tokenizer_mc, model_emb)
    print(f"{name} x {emb_model.split('/')[-1]:<15}: {acc:.2f}")

Some weights of RobertaForMultipleChoice were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ROBERTA x all-mpnet-base-v2: 0.38


Some weights of LukeForMultipleChoice were not initialized from the model checkpoint at studio-ousia/luke-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


LUKE x all-MiniLM-L6-v2: 0.17


Some weights of AlbertForMultipleChoice were not initialized from the model checkpoint at albert-base-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ALBERT x multi-qa-mpnet-base-dot-v1: 0.26


Some weights of YosoForMultipleChoice were not initialized from the model checkpoint at uw-madison/yoso-4096 and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


YOSO x bert-base-nli-mean-tokens: 0.42


Some weights of DebertaV2ForMultipleChoice were not initialized from the model checkpoint at microsoft/deberta-v2-xlarge and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DEBERTA x LaBSE          : 0.21


Δοκιμάζοντας όλα τα προηγούμενα προεκπαιδευμένα μοντέλα σε multiple choice downstream tasks, συνδυάζοντας τα κατάλληλα με sentence transformers, παρατηρούμε λίγο καλύτερα αποτελέσματα, αλλά παραμένουν χειρότερα από το προηγούμενο task. Η καλύτερη ακρίβεια που επιτυγχάνεται είναι από το συνδυασμό YOSOxbert-base-nli-mean-tokens (0.42), ενώ ο συνδυασμός LUKExall-MiniLM-L6-v2 επιτυγχάνει ακρίβεια μόνο 0.17. Τα αποτελέσματα αυτά οφείλονται στους λόγους που αναφέρθηκαν προηγουμένως.

## Β3. Winogrande dataset

Το [Winogrande dataset](https://huggingface.co/datasets/winogrande) αποτελείται από προτάσεις που μία λέξη τους έχει αφαιρεθεί και δίνονται δύο πιθανές επιλογές συμπλήρωσης του κενού. Για παράδειγμα, δοθείσας της πρότασης "John moved the couch from the garage to the backyard to create space. The _ is small.", υπάρχουν δύο πιθανές εναλλακτικές:

- "garage"
- "backyard"

Η δυσκολία της συμπλήρωσης έγκειται στο ότι και οι δύο λέξεις αναφέρονται στην πρόταση, οπότε το μοντέλο θα πρέπει να διαθέτει υψηλές δυνατότητες κατανόησης γλώσσας προκειμένου να επιλέξει μια νοηματικά σωστή συμπλήρωση.

Για λόγους επιτάχυνσης, επιλέξτε ένα τυχαίο υποσύνολο 100 δειγμάτων από το training set του Winogrande.


In [ ]:
# insert your code here (load dataset)

dataset_raw = load_dataset("winogrande", "winogrande_debiased", split = ["train[0:100]"])

Με κατάλληλο [μετασχηματισμό](https://huggingface.co/DeepPavlov/roberta-large-winogrande) της παραπάνω εισόδου (πρόταση με κενό και δύο επιλογές συμπλήρωσης), καλείστε να καταγράψετε το accuracy σχετικών μοντέλων που επιλύουν το πρόβλημα, συγκρίνοντας το predicted label με το πραγματικό label (1: πρώτη επιλογή, 2: δεύτερη επιλογή). Ουσιαστικά θα πρέπει να αναγάγετε το παραπάνω πρόβλημα σε κάποιο πιο κλασικό πρόβλημα της επεξεργασίας φυσικής γλώσσας.

Δοκιμάστε τουλάχιστον 3 κατάλληλα μοντέλα από το Huggingface για να προσεγγίσετε το πρόβλημα του Winogrande. Προτείνουμε τη χρήση pipelines για τη διευκόλυνσή σας.

In [ ]:
# insert your code here (load models)

dataset = [{"sentence": d["sentence"].split("_")[0], "choice1": d["option1"] + d["sentence"].split("_")[1], \
            "choice2": d["option2"] + d["sentence"].split("_")[1], "answer": int(d["answer"])} for d in dataset_raw[0]]

In [ ]:
# insert your code here (create pipelines)

from transformers import pipeline

In [ ]:
# insert your code here (function for predicting best fill)

def get_accuracy(model):
  oracle = pipeline("zero-shot-classification", model=model)
  count = 0
  for i in range(100):
    d = dataset[i]
    res = oracle(d["sentence"], candidate_labels=[d["choice1"], d["choice2"]])["labels"][0]
    if res == d["choice1"]:        #labels[0] contains the best matching label
      choice = 1
    else:
      choice = 2

    if choice == d["answer"]:
      count += 1

  return count / 100

print("Bart Large")
print(get_accuracy("facebook/bart-large-mnli"))

print("Roberta Base")
print(get_accuracy("cross-encoder/nli-roberta-base"))

print("Pavlov")
print(get_accuracy("DeepPavlov/roberta-large-winogrande"))

Bart Large


Device set to use cuda:0


0.56
Roberta Base


Device set to use cuda:0


0.51
Pavlov


Device set to use cuda:0
Failed to determine 'entailment' label id from the label2id mapping in the model config. Setting to -1. Define a descriptive label2id mapping in the model config to ensure correct outputs.


0.99


Για να αντιμετωπίσουμε το πρόβλημα της συμπλήρωσης κενού με δύο λέξεις ως επιλογές, οι οποίες αναφέρονται και οι δύο μέσα στην πρόταση, θα μετασχηματίσουμε κατάλληλα το σύνολο των δεδομένων μας. Συγκεκριμένα, θα σπάσουμε την πρόταση στην κάτω παύλα, θα θεωρήσουμε την πρόταση πρίν από την κάτω παύλα ως την πρώτη πρόταση και θα έχουμε ως πιθανές επιλογές δύο προτάσεις, την πρώτη από τις διαθέσιμες λέξεις μαζί με την υπόλοιπη πρόταση και τη δεύτερη από τις διαθέσιμες λέξεις μαζί με την υπόλοιπη πρόταση. Έτσι, μετασχηματίζουμε το πρόβλημα σε ένα zero shot classification task, στο οποίο δίνεται μία πρόταση και πρέπει να ταξονιμηθεί σε κλάσεις τις οποίες το μονέλο δεν έχει ξαναδεί. Εξετάζοντας 3 μοντέλα σε αυτή την άσκηση, παρατηρούμε πως γενικά η ακρίβεια είναι πολύ καλύτερη από τις προηγούμενες ασκήσεις (0.5), το οποίο είναι αναμενόμενο, αφού απαιτούνται μόνο γενικές γλωσσικές γνώσεις και όχι εξειδίκευση σε συγκεκριμένο γλωσσικό task. Το τελευταίο μοντέλο παρουσιάζει τη σχεδόν τέλεια ακρίβεια 0.99, αλλά έχει εκπαιδευτεί στο winograde dataset, επομένως παρουσιάζει overfitting.